In [84]:
import pandas as pd
import numpy as np

In [85]:
item = pd.read_csv("/Users/trba/Documents/Projects/Serbia_housing/nekretnine_rs_raw.csv")
item_halo =pd.read_csv("/Users/trba/Documents/Projects/Serbia_housing/halo_oglasi_raw.csv")

In [86]:
#Ovo je sredjivanje titla

item["title"] = item["title"].str.replace(r"\d+|EUR|€/m²|€|m²|m2|\.", "", regex=True).str.strip()
item["title"] = item['title'].str.replace(r", ,","",regex = True).str.strip()
item["title"] = item['title'].str.strip(",")
item['title'].head()



0                Stari grad, Dorćol, Strahinjića Bana 
1    Vracar, Metohijska uknjizen, u odlicnom stanju...
2         Rakovica, Petlovo brdo, Dobrivoja Petkovića 
3    Izuzetna novogradnja kod TC BIG , komforan  du...
4    Vanvremenski salonac sa baštom i garažom – src...
Name: title, dtype: str

In [87]:
#Zavrseno sredjivanje _price_total
item["price_total"] = (
    item["price_total"]
    .str.split("\n").str[0]
    .str.replace(r"EUR|eur|€|RSD|rsd|DIN|din|Evra|evra|Eura|eura", "", regex=True)
    .str.replace(r"[.,\s]", "", regex=True)
    .str.replace(r"od ", "", regex=True)
    .str.strip()
    .astype("Int64")
)

In [88]:
item['price_total'].head()

0    319000
1    308000
2    160000
3    266000
4    448000
Name: price_total, dtype: Int64

In [89]:
#Zavrseno sredjivanje _price_total
item["price_per_m2"] = (
    item["price_per_m2"]
    .str.split("\n").str[0]
    .str.replace(r"EUR|€/m²|€|m²|m2|\.", "", regex=True).str.strip()
    .str.replace(r"[.,\s]", "", regex=True)
    .str.replace(r"od ", "", regex=True)
    .str.strip()
    .astype("Int64")
)

In [90]:
item["price_per_m2"].head()

0    4908
1    3500
2    2388
3    2418
4    4480
Name: price_per_m2, dtype: Int64

In [91]:
np.unique(item["Tip nekretnine"])

array(['Dvosoban stan', 'Garsonjera', 'Jednosoban stan', 'Trosoban stan',
       'Četvorosoban stan'], dtype=object)

In [92]:
## Zavrsen tip nekretnine
item["Tip nekretnine"] = np.where(
    item["Tip nekretnine"].str.lower().str.contains("stan",na = False),
    "stan",
        np.where(item["Tip nekretnine"].str.lower().str.contains("kuca|kuća",na = False),
            "kuća",
    None
))


In [93]:
#Kvadratura zavrsena
item["Kvadratura"] = item["Kvadratura"].str.replace(r"m2|m²|kvadrata|~|oko|","",regex = True).str.strip()
item["Kvadratura"] = item["Kvadratura"].str.replace(r",",".",regex = True).str.strip()
item["Kvadratura"] = item["Kvadratura"].astype(float)

In [94]:
item['Kvadratura'].head()

0     65.0
1     88.0
2     67.0
3    110.0
4    100.0
Name: Kvadratura, dtype: float64

In [95]:
#Broj soba promenjeno u float za iposobne stanove
item['Broj soba'] = item['Broj soba'].astype(float)

In [96]:
#Oglasivac mora iz rich teksta. To cu kasnije

In [97]:
item_halo['Tip objekta'].unique()

<StringArray>
['Stara gradnja', 'Novogradnja', nan]
Length: 3, dtype: str

In [98]:
item['Stanje objekta'].unique()

<StringArray>
[          'Izvorno stanje',                      'Lux',
 'Kompletna rekonstrukcija',              'U izgradnji',
              'Novogradnja',       'Standardna gradnja']
Length: 6, dtype: str

In [99]:
item['Tip objekta'].unique()

array([nan])

In [100]:
item_halo['Stanje objekta'].unique()

<StringArray>
['Renovirano', 'Izvorno stanje', nan, 'Lux']
Length: 4, dtype: str

In [101]:
#Ovo treba na vrhu da stoji
item = item.replace('Nan', np.nan)

In [102]:
#Zato sto je pre bilo float ???
item['Tip objekta'] = item['Tip objekta'].astype('object')

In [103]:
#Tip objekta je gotov sa ovime
item.loc[
    item['Tip objekta'].isna() &
    item['Stanje objekta'].isin(['Kompletna rekonstrukcija','Renovirano']),
    'Tip objekta'
] = 'Stara_gradnja'

item.loc[
    item['Tip objekta'].isna() &
    item['Stanje objekta'].isin(['Novo', 'Lux']),
    'Tip objekta'
] = 'Novo_gradnja'

In [104]:
item['Tip objekta'].head()

0              NaN
1              NaN
2              NaN
3     Novo_gradnja
4    Stara_gradnja
Name: Tip objekta, dtype: object

In [105]:
item['Tip objekta'] = item['Tip objekta'].replace({
    'Stara gradnje': 'Stara_gradnja',
    'Novogradnja': 'Novo_gradnja'
})

In [106]:
#Stanje objekta je jedino bilo lose
item['Stanje objekta'] = item['Stanje objekta'].replace('Izvorno stanje','Izvorno_stanje')

In [107]:
item.columns

Index(['url', 'title', 'price_total', 'price_per_m2', 'Tip nekretnine',
       'Kvadratura', 'Broj soba', 'Oglašivač', 'Tip objekta', 'Stanje objekta',
       'Grejanje', 'Sprat', 'Ukupna spratnost', 'Uknjižen', 'Terasa',
       'Telefon', 'Interfon', 'Klima', 'Video nadzor', 'Topla voda',
       'Internet', 'Parking', 'Garaža', 'Lift', 'Podrum',
       'Linije gradskog prevoza', 'Dodatni opis'],
      dtype='str')

In [108]:
item['Grejanje'].unique()

<StringArray>
[                                                     'Grejanje: Klima uređaj, Ostalo',
                                                        'Grejanje: Centralno grejanje',
                                          'Grejanje: Centralno grejanje, Klima uređaj',
                                            'Grejanje: Podno grejanje, Toplotna pumpa',
 'Grejanje: Klima uređaj, Etažno grejanje na struju, Etažno grejanje na čvrsto gorivo',
                                                    'Grejanje: Etažno grejanje na gas',
                                                                                   nan,
                                                 'Grejanje: Etažno grejanje na struju',
                                                                    'Grejanje: Ostalo']
Length: 9, dtype: str

In [109]:
#grejanje je zavrseno sa ovime
item['Grejanje'] = (
    item['Grejanje']
    .astype('string')
    .str.replace(r"Grejanje:\s*", "", regex=True)
    .str.replace(r"\bKlima uređaj\b|\bKlima uredjaj\b", "Klima", regex=True)
    .str.replace(r"\bOstalo\b", "", regex=True)
    .str.replace(r"\s*,\s*", ", ", regex=True)
    .str.replace(r"^,\s*|\s*,$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [110]:
item['Grejanje'] = (
    item['Grejanje']
    .str.replace(r"\bCentralno grejanje\b", "centralno", regex=True)
    .str.replace(r"\bPodno grejanje\b", "podno", regex=True)
    .str.replace(r"\bToplotna pumpa\b", "toplotna_pumpa", regex=True)
    .str.replace(r"\bEtažno grejanje na struju\b", "etazno_struja", regex=True)
    .str.replace(r"\bEtažno grejanje na gas\b", "etazno_gas", regex=True)
    .str.replace(r"\bEtažno grejanje na čvrsto gorivo\b|\bEtažno grejanje na cvrsto gorivo\b", "etazno_cvrsto_gorivo", regex=True)
    .str.replace(r"\bKlima\b", "klima", regex=True)
)

In [111]:
item['Grejanje'] = item['Grejanje'].str.replace(r",\s*$", "", regex=True)

In [112]:
item['Grejanje'].head()

0                                         klima
1                                     centralno
2                              centralno, klima
3                         podno, toplotna_pumpa
4    klima, etazno_struja, etazno_cvrsto_gorivo
Name: Grejanje, dtype: string

In [113]:
item['Grejanje'].unique()

<StringArray>
[                                     'klima',
                                  'centralno',
                           'centralno, klima',
                      'podno, toplotna_pumpa',
 'klima, etazno_struja, etazno_cvrsto_gorivo',
                                 'etazno_gas',
                                         <NA>,
                              'etazno_struja',
                                           '']
Length: 9, dtype: string

In [114]:
item['Sprat'].unique()

<StringArray>
[               '3',                '6',                '2',
 'Visoko prizemlje',                '7',                nan,
                '4',        'Prizemlje',                '9',
                '5',                '8',                '1']
Length: 12, dtype: str

In [115]:
#Sprat zavrsen
item['Sprat'] = item['Sprat'].str.replace({
    'Visoko prizemlje': '0.5',
    'Prizemlje': '0',
    'Suturen': '-0.5'
})
item['Sprat'] = pd.to_numeric(item['Sprat'], errors='coerce')

In [116]:
item['Sprat'].unique()

array([3. , 6. , 2. , 0.5, 7. , nan, 4. , 0. , 9. , 5. , 8. , 1. ])

In [117]:
item['Ukupna spratnost'].head()

0    NaN
1    7.0
2    NaN
3    7.0
4    2.0
Name: Ukupna spratnost, dtype: float64

In [118]:
#Spratnost zavrsena

item['Ukupna spratnost'] = (
    item['Ukupna spratnost']
    .astype('string')
    .str.strip()
    .str.extract(r'(\d+)', expand=False)
)

item['Ukupna spratnost'] = pd.to_numeric(item['Ukupna spratnost'], errors='coerce')

In [119]:
item['Ukupna spratnost'].head()

0    <NA>
1       7
2    <NA>
3       7
4       2
Name: Ukupna spratnost, dtype: Int64

In [120]:
item.columns

Index(['url', 'title', 'price_total', 'price_per_m2', 'Tip nekretnine',
       'Kvadratura', 'Broj soba', 'Oglašivač', 'Tip objekta', 'Stanje objekta',
       'Grejanje', 'Sprat', 'Ukupna spratnost', 'Uknjižen', 'Terasa',
       'Telefon', 'Interfon', 'Klima', 'Video nadzor', 'Topla voda',
       'Internet', 'Parking', 'Garaža', 'Lift', 'Podrum',
       'Linije gradskog prevoza', 'Dodatni opis'],
      dtype='str')

In [121]:
item_halo['Uknjižen'].unique()

<StringArray>
['Da', nan]
Length: 2, dtype: str

In [122]:
# Ovo je za Ukljizen i naredni red
item['Uknjižen'] = item['Uknjižen'].replace({
    'Da': 1,
    'Ne': 0})
item['Uknjižen'] = pd.to_numeric(item['Uknjižen'], errors='coerce')

In [123]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'nije\s+uknjižen|nije\s+uknjizen|'
    r'biće\s+uknjižen|bice\s+uknjizen|'
    r'uskoro\s+uknjižen|uskoro\s+uknjizen|'
    r'u\s+procesu\s+uknjiženja|u\s+procesu\s+uknjizenja|'
    r'predat[ao]?\s+za\s+uknjiženje|predat[ao]?\s+za\s+uknjizenje|'
    r'može\s+uknjiženje|moze\s+uknjizenje|'
    r'zgrada\s+je\s+uknjižena|zgrada\s+je\s+uknjizena|'
    r'objekat\s+je\s+uknjižen|objekat\s+je\s+uknjizen',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'uknjižen\s+na|uknjizen\s+na|'
    r'uknjižena\s+površina|uknjizena\s+povrsina|'
    r'uknjiženo\s+\d+|uknjizeno\s+\d+|'
    r'stan\s+je\s+uknjižen|stan\s+je\s+uknjizen|'
    r'uknjižen\s+je|uknjizen\s+je|'
    r'uknjižena\s+je|uknjizena\s+je|'
    r'\buknjižen\b|\buknjizen\b',
    regex=True,
    na=False
)

item.loc[
    item['Uknjižen'].isna() & pos_mask & ~neg_mask,
    'Uknjižen'
] = 1

In [124]:
item['Uknjižen'].unique()

array([ 1., nan])

In [125]:
# Ovo je za Terasu
item['Terasa'] = item['Terasa'].replace({
    'Da': 1,
    'Ne': 0})
item['Terasa'] = pd.to_numeric(item['Terasa'], errors='coerce')

In [126]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+terase|bez\s+terasa|nema\s+terasu|nema\s+terase|bez\s+lođe|bez\s+lođa|bez\s+lodje|bez\s+lodja',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\bterasa\b|\bterase\b|\blođa\b|\blođe\b|\blodja\b|\blodje\b',
    regex=True,
    na=False
)

item.loc[
    item['Terasa'].isna() & pos_mask & ~neg_mask,
    'Terasa'
] = 1

item.loc[
    item['Terasa'].isna() & neg_mask,
    'Terasa'
] = 0

In [127]:
item['Terasa'] = item['Terasa'].map({1: True, 0: False}).astype('boolean')

In [128]:
item['Terasa'].head()

0    True
1    True
2    True
3    True
4    True
Name: Terasa, dtype: boolean

In [129]:
item['Dodatni opis'].head()

0    Agencijski ID: 94045, III/VI, klima, lift, sal...
1    Na prodaju odlican trosoban stan (lako adaptib...
2    Agencijski ID: 125109, II/IV, klima, uknjižen,...
3    Objekat ima upotrebnu dozvolu i zgrada je uknj...
4    U ponudi je jedinstven salonski stan u repreze...
Name: Dodatni opis, dtype: str

In [130]:
#Sredjivanje Dodatni opis - ovo isto treba mnogo ranije da se uradi jer se odavde vade featursi
item['Dodatni opis'] = (
    item['Dodatni opis']
    .astype('string')
    # linkovi
    .str.replace(r'http\S+|www\.\S+', '', regex=True)
    # procenti, npr. 2%, 3.5 %
    .str.replace(r'\b\d+(?:[.,]\d+)?\s*%', '', regex=True)
    # brojevi telefona: 060/123-4567, 011/123-456, +381 60 123 45 67 itd.
    .str.replace(r'(?:tel(?:efon)?i?\s*:?\s*)?(?:\+381|0)\s*\d(?:[\d\s/\-•()]*)\d', '', regex=True)
    .str.replace(r'Više informacija na:\s*', '', regex=True)
    .str.replace(r'Agencijska provizija\.?\s*', '', regex=True)
    .str.replace(r'Telefoni?:\s*', '', regex=True)
    .str.replace(r'Kontakt van radnog vremena:\s*', '', regex=True)
    .str.replace(r'\b[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}\b', '', regex=True)
    # višestruki razmaci
    .str.replace(r'\s+', ' ', regex=True)
    # razmaci oko tačke i zareza
    .str.replace(r'\s+([,.:;])', r'\1', regex=True)
    .str.strip()
)

In [131]:
item['Dodatni opis'].head()

0    Agencijski ID: 94, III/VI, klima, lift, salons...
1    Na prodaju odlican trosoban stan (lako adaptib...
2    Agencijski ID: 125109, II/IV, klima, uknjižen,...
3    Objekat ima upotrebnu dozvolu i zgrada je uknj...
4    U ponudi je jedinstven salonski stan u repreze...
Name: Dodatni opis, dtype: string

In [132]:
item.loc[2, 'Dodatni opis']

'Agencijski ID: 125109, II/IV, klima, uknjižen, izvorno stanje, ulaz dobar, stara gradnja, fasada dobra, parking, podrum. Izuzetno lep stan u porodicom, mirnom kraju na Petlovom brdu. Stan je korisne površine 57m2 + dve lođe ukupno 12.4m2 + podrum. Sastoji se od spavaće sobe, dnevne sobe, trpezarije, kupatila, kuhinje, predsoblja, hodnika, dve lođe od kojih je jedna zastakljena. Stan se nalazi na drugom spratu zgrade koja ima 4 sprata, nema lift. Izvorno stanje, kvalitetna starogradnja. Vlasništvo 1/1, odmah useljiv...'

In [133]:
#'Telefon', 'Interfon', 'Klima', 'Video nadzor', 'Topla voda',
       #'Internet', 'Parking', 'Garaža', 'Lift', 'Podrum',
       #'Linije gradskog prevoza',



#Ovo fali

In [134]:
#Interfon
item['Interfon'] = item['Interfon'].map({'Da': True, 'Ne': False}).astype('boolean')

In [135]:
item['Interfon'].head()

0    <NA>
1    <NA>
2    <NA>
3    <NA>
4    <NA>
Name: Interfon, dtype: boolean

In [136]:
item.loc[
    item['Interfon'].isna() &
    (item['Tip objekta'] == 'Novo_gradnja'),
    'Interfon'
] = True

In [137]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+interfona|nema\s+interfon|bez\s+interfonske\s+veze',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\binterfon\b|\binterfonsk\w*\b|\bvideo\s+interfon\b',
    regex=True,
    na=False
)

item.loc[
    item['Interfon'].isna() & pos_mask & ~neg_mask,
    'Interfon'
] = True

item.loc[
    item['Interfon'].isna() & neg_mask,
    'Interfon'
] = False

In [138]:
item['Interfon'].head()

0    <NA>
1    <NA>
2    <NA>
3    True
4    <NA>
Name: Interfon, dtype: boolean

In [139]:
#Klima
item['Klima'] = item['Klima'].map({'Da': True, 'Ne': False}).astype('boolean')

In [140]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+klime|nema\s+klimu|nije\s+ugrađena\s+klima|nije\s+ugradjena\s+klima',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\bklima\b|\bklime\b|\bklima uređaj\b|\bklima uredjaj\b|\bklimatizovan\b|\bklimatizirana\b',
    regex=True,
    na=False
)

item.loc[
    item['Klima'].isna() & pos_mask & ~neg_mask,
    'Klima'
] = True

item.loc[
    item['Klima'].isna() & neg_mask,
    'Klima'
] = False

In [141]:
item['Klima'].head()

0    True
1    <NA>
2    True
3    <NA>
4    True
Name: Klima, dtype: boolean

In [142]:
#Video nadzor
#Klima
item['Video nadzor'] = item['Video nadzor'].map({'Da': True, 'Ne': False}).astype('boolean')

In [143]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+video\s+nadzora|'
    r'nema\s+video\s+nadzor|'
    r'bez\s+nadzornih\s+kamera|'
    r'nema\s+kamere|'
    r'bez\s+kamere|'
    r'bez\s+nadzora',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\bvideo\s*nadzor\b|'
    r'\bvideo-nadzor\b|'
    r'\bnadzorne?\s+kamere\b|'
    r'\bkamere\b|'
    r'\bkamera\b',
    regex=True,
    na=False
)

item.loc[
    item['Video nadzor'].isna() & pos_mask & ~neg_mask,
    'Video nadzor'
] = True

item.loc[
    item['Video nadzor'].isna() & neg_mask,
    'Video nadzor'
] = False

In [144]:
item['Video nadzor'].head()

0    <NA>
1    <NA>
2    <NA>
3    True
4    <NA>
Name: Video nadzor, dtype: boolean

In [145]:
item['Internet'].unique()

<StringArray>
[nan, 'da']
Length: 2, dtype: str

In [146]:
#Internet
item['Internet'] = item['Internet'].map({'da': True, 'ne': False}).astype('boolean')

In [147]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+interneta|nema\s+internet|bez\s+internet priključka|bez\s+internet prikljucka',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\binternet\b|\bwi[\s\-]?fi\b|\boptički internet\b|\bopticki internet\b|\bkablovski internet\b',
    regex=True,
    na=False
)

item.loc[
    item['Internet'].isna() & pos_mask & ~neg_mask,
    'Internet'
] = True

item.loc[
    item['Internet'].isna() & neg_mask,
    'Internet'
] = False

In [148]:
item['Internet'].head()

0    <NA>
1    <NA>
2    <NA>
3    True
4    True
Name: Internet, dtype: boolean

In [149]:
item['Parking'].unique()

<StringArray>
[nan, 'da']
Length: 2, dtype: str

In [150]:
#parking
item['Parking'] = item['Parking'].map({'da': True,'ne': False}).astype('boolean')

In [151]:
item['Parking'].unique()

<BooleanArray>
[<NA>, True]
Length: 2, dtype: boolean

In [152]:

opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+parkinga|'
    r'nema\s+parking|'
    r'bez\s+parking mesta|'
    r'bez\s+parking mesta|'
    r'bez\s+parkinga\s+i\s+garaže|'
    r'bez\s+parkinga\s+i\s+garaze',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\bparking\b|'
    r'\bparking mesto\b|'
    r'\bparking mesta\b|'
    r'\bparking prostor\b|'
    r'\bparking uz zgradu\b|'
    r'\bobezbeđen parking\b|'
    r'\bobezbedjen parking\b|'
    r'\bslobodan parking\b|'
    r'\bparking ispred zgrade\b',
    regex=True,
    na=False
)

item.loc[
    item['Parking'].isna() & pos_mask & ~neg_mask,
    'Parking'
] = True

item.loc[
    item['Parking'].isna() & neg_mask,
    'Parking'
] = False

In [153]:
item['Parking'].unique()

<BooleanArray>
[<NA>, True]
Length: 2, dtype: boolean

In [154]:
item['Garaža'].unique()

<StringArray>
[nan, 'da']
Length: 2, dtype: str

In [155]:
item['Garaža'] = item['Garaža'].map({'da': True,'ne':False}).astype('boolean')

In [156]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+garaže|'
    r'bez\s+garaze|'
    r'nema\s+garažu|'
    r'nema\s+garazu|'
    r'bez\s+garažnog\s+mesta|'
    r'bez\s+garaznog\s+mesta',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\bgaraža\b|'
    r'\bgaraza\b|'
    r'\bgaražno\s+mesto\b|'
    r'\bgarazno\s+mesto\b|'
    r'\bgaraža\s+u\s+zgradi\b|'
    r'\bgaraza\s+u\s+zgradi\b|'
    r'\bgaražno\s+mesto\b|'
    r'\bgarazno\s+mesto\b',
    regex=True,
    na=False
)

item.loc[
    item['Garaža'].isna() & pos_mask & ~neg_mask,
    'Garaža'
] = True

item.loc[
    item['Garaža'].isna() & neg_mask,
    'Garaža'
] = False

In [157]:
item['Garaža'].head()

0    <NA>
1    <NA>
2    <NA>
3    <NA>
4    True
Name: Garaža, dtype: boolean

In [158]:
item['Lift'].unique()

<StringArray>
['da', nan]
Length: 2, dtype: str

In [159]:
item['Lift'] = item['Lift'].map({'da': True,'ne':False}).astype('boolean')

In [160]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+lifta|'
    r'nema\s+lift',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\blift\b|\bliftom\b|\blifta\b',
    regex=True,
    na=False
)

item.loc[
    item['Lift'].isna() & pos_mask & ~neg_mask,
    'Lift'
] = True

item.loc[
    item['Lift'].isna() & neg_mask,
    'Lift'
] = False

In [161]:
item['Podrum'] = item['Podrum'].map({'da': True,'ne':False}).astype('boolean')

In [162]:
opis = item['Dodatni opis'].astype('string').str.lower()

neg_mask = opis.str.contains(
    r'bez\s+podruma|'
    r'nema\s+podrum|'
    r'bez\s+pripadajućeg\s+podruma|'
    r'bez\s+pripadajuceg\s+podruma',
    regex=True,
    na=False
)

pos_mask = opis.str.contains(
    r'\bpodrum\b|'
    r'\bpodrumska\s+prostorija\b|'
    r'\bpripadajući\s+podrum\b|'
    r'\bpripadajuci\s+podrum\b',
    regex=True,
    na=False
)

item.loc[
    item['Podrum'].isna() & pos_mask & ~neg_mask,
    'Podrum'
] = True

item.loc[
    item['Podrum'].isna() & neg_mask,
    'Podrum'
] = False

In [163]:
item['Topla voda'] = item['Topla voda'].astype('boolean')

In [164]:
item.head()

,url,title,price_total,price_per_m2,Tip nekretnine,Kvadratura,Broj soba,Oglašivač,Tip objekta,Stanje objekta,...,Klima,Video nadzor,Topla voda,Internet,Parking,Garaža,Lift,Podrum,Linije gradskog prevoza,Dodatni opis
0,https://www.nekretnine.rs/stambeni-objekti/sta...,"Stari grad, Dorćol, Strahinjića Bana",319000,4908,stan,65.0,3.0,NaN,NaN,Izvorno_stanje,...,True,<NA>,<NA>,<NA>,<NA>,<NA>,True,True,NaN,"Agencijski ID: 94, III/VI, klima, lift, salons..."
1,https://www.nekretnine.rs/stambeni-objekti/sta...,"Vracar, Metohijska uknjizen, u odlicnom stanju...",308000,3500,stan,88.0,3.0,NaN,NaN,Izvorno_stanje,...,<NA>,<NA>,<NA>,<NA>,True,<NA>,True,True,NaN,Na prodaju odlican trosoban stan (lako adaptib...
2,https://www.nekretnine.rs/stambeni-objekti/sta...,"Rakovica, Petlovo brdo, Dobrivoja Petkovića",160000,2388,stan,67.0,2.0,NaN,NaN,Izvorno_stanje,...,True,<NA>,<NA>,<NA>,True,<NA>,False,True,NaN,"Agencijski ID: 125109, II/IV, klima, uknjižen,..."
3,https://www.nekretnine.rs/stambeni-objekti/sta...,"Izuzetna novogradnja kod TC BIG , komforan du...",266000,2418,stan,110.0,4.0,NaN,Novo_gradnja,Lux,...,<NA>,True,<NA>,True,True,<NA>,True,<NA>,NaN,Objekat ima upotrebnu dozvolu i zgrada je uknj...
4,https://www.nekretnine.rs/stambeni-objekti/sta...,Vanvremenski salonac sa baštom i garažom – src...,448000,4480,stan,100.0,4.0,NaN,Stara_gradnja,Kompletna rekonstrukcija,...,True,<NA>,<NA>,True,<NA>,True,<NA>,True,NaN,U ponudi je jedinstven salonski stan u repreze...
